In [2]:
import pandas as pd
from sqlalchemy import create_engine, text
from dotenv import load_dotenv, find_dotenv
import os

# Load .env
load_dotenv(find_dotenv())

DB_URL = os.getenv("LOCAL_DATABASE_URL")

if DB_URL and DB_URL.startswith("postgresql://"):
    DB_URL = DB_URL.replace(
        "postgresql://",
        "postgresql+psycopg2://",
        1
    )

engine = create_engine(DB_URL)

print("Database connection berhasil")
print(f"Target: {DB_URL.split('@')[1] if DB_URL else 'NONE'}")

Database connection berhasil
Target: localhost:5432/retail_analytics


In [5]:
silver_columns = pd.read_sql(
    """
    SELECT 
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver'
    ORDER BY table_name, ordinal_position
    """,
    engine
)

order_tables = silver_columns[
    silver_columns["table_name"].isin([
        "orders",
        "order_items",
        "order_promotions",
        "payment_events",
        "refund_events",
        "return_events"
    ])
]

display(order_tables)

,table_name,column_name,data_type
40,order_items,order_id,text
41,order_items,quantity,bigint
42,order_items,product_id,text
43,order_items,unit_price,double precision
44,order_items,order_item_id,text
45,order_items,source_row_id,text
46,order_items,item_discount_amount,double precision
47,order_items,ingested_at_utc,timestamp with time zone
48,order_promotions,order_id,text
49,order_promotions,promotion_id,text


In [7]:
customer_tables = pd.read_sql(
    """
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver'
      AND table_name IN (
          'customers',
          'customer_profiles',
          'customer_addresses'
      )
    ORDER BY table_name, ordinal_position
    """,
    engine
)

display(customer_tables)

,table_name,column_name,data_type
0,customer_addresses,city_id,text
1,customer_addresses,address_id,text
2,customer_addresses,customer_id,text
3,customer_addresses,valid_to_utc,text
4,customer_addresses,source_row_id,text
5,customer_addresses,valid_from_utc,text
6,customer_addresses,ingested_at_utc,timestamp with time zone
7,customer_profiles,city_id,text
8,customer_profiles,customer_id,text
9,customer_profiles,valid_to_utc,text


In [8]:
customers_check = pd.read_sql(
    """
    SELECT *
    FROM silver.customers
    LIMIT 10
    """,
    engine
)

display(customers_check)

,city_id,customer_id,source_row_id,created_at_utc,customer_segment,ingested_at_utc
0,JKT,CUST-00001,CUSTOMER-ROW-000001,2025-07-29T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
1,MDN,CUST-00002,CUSTOMER-ROW-000002,2026-02-01T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
2,BDG,CUST-00003,CUSTOMER-ROW-000003,2026-04-11T00:00:00Z,enterprise,2026-09-22 20:52:04.880993+00:00
3,BDG,CUST-00004,CUSTOMER-ROW-000004,2025-07-10T00:00:00Z,enterprise,2026-09-22 20:52:04.880993+00:00
4,JKT,CUST-00005,CUSTOMER-ROW-000005,2025-08-23T00:00:00Z,small_business,2026-09-22 20:52:04.880993+00:00
5,MDN,CUST-00006,CUSTOMER-ROW-000006,2026-06-06T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
6,JKT,CUST-00007,CUSTOMER-ROW-000007,2026-02-22T00:00:00Z,enterprise,2026-09-22 20:52:04.880993+00:00
7,BPN,CUST-00008,CUSTOMER-ROW-000008,2025-09-07T00:00:00Z,consumer,2026-09-22 20:52:04.880993+00:00
8,DPS,CUST-00009,CUSTOMER-ROW-000009,2026-03-01T00:00:00Z,small_business,2026-09-22 20:52:04.880993+00:00
9,MKS,CUST-00010,CUSTOMER-ROW-000010,2025-06-29T00:00:00Z,small_business,2026-09-22 20:52:04.880993+00:00


In [9]:
customer_duplicates = pd.read_sql(
    """
    SELECT
        customer_id,
        COUNT(*) AS row_count
    FROM silver.customers
    GROUP BY customer_id
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    """,
    engine
)

display(customer_duplicates)

,customer_id,row_count


In [10]:
orders_customer_test = pd.read_sql(
    """
    SELECT
        o.order_id,
        o.customer_id,
        c.city_id,
        c.customer_segment
    FROM (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    ) o
    LEFT JOIN silver.customers c
        ON o.customer_id = c.customer_id
    ORDER BY o.order_id
    LIMIT 10
    """,
    engine
)

display(orders_customer_test)

,order_id,customer_id,city_id,customer_segment
0,ORD-000001,CUST-01757,BPN,enterprise
1,ORD-000002,CUST-01160,DPS,small_business
2,ORD-000003,CUST-00962,BDG,enterprise
3,ORD-000004,CUST-00181,DPS,consumer
4,ORD-000005,CUST-02298,MKS,enterprise
5,ORD-000006,CUST-00630,SBY,enterprise
6,ORD-000007,CUST-00632,DPS,consumer
7,ORD-000008,CUST-01792,BPN,enterprise
8,ORD-000009,CUST-01220,BPN,enterprise
9,ORD-000010,CUST-00459,BPN,consumer


In [11]:
first_order_test = pd.read_sql(
    """
    SELECT
        order_id,
        customer_id,
        ordered_at_utc,
        CASE
            WHEN ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY ordered_at_utc, order_id
            ) = 1
            THEN TRUE
            ELSE FALSE
        END AS first_order_flag
    FROM (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    ) o
    ORDER BY customer_id, ordered_at_utc
    LIMIT 20
    """,
    engine
)

display(first_order_test)

,order_id,customer_id,ordered_at_utc,first_order_flag
0,ORD-006860,CUST-00001,2026-06-26T15:18:00Z,True
1,ORD-009732,CUST-00001,2026-07-18T01:09:00Z,False
2,ORD-002644,CUST-00001,2026-08-12T23:07:00Z,False
3,ORD-006550,CUST-00001,2026-08-19T06:24:00Z,False
4,ORD-009265,CUST-00001,2026-09-02T02:45:00Z,False
5,ORD-009718,CUST-00002,2026-06-29T21:48:00Z,True
6,ORD-002388,CUST-00002,2026-08-25T02:09:00Z,False
7,ORD-007417,CUST-00002,2026-08-25T20:39:00Z,False
8,ORD-004568,CUST-00002,2026-08-27T02:03:00Z,False
9,ORD-003715,CUST-00003,2026-06-24T15:01:00Z,True


In [12]:
product_tables = pd.read_sql(
    """
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver'
      AND table_name IN (
          'products',
          'product_categories'
      )
    ORDER BY table_name, ordinal_position
    """,
    engine
)

display(product_tables)

,table_name,column_name,data_type
0,product_categories,product_id,text
1,product_categories,category_id,text
2,product_categories,valid_to_utc,text
3,product_categories,category_name,text
4,product_categories,source_row_id,text
5,product_categories,valid_from_utc,text
6,product_categories,ingested_at_utc,timestamp with time zone
7,products,sku,text
8,products,product_id,text
9,products,unit_price,double precision


In [13]:
product_category_duplicates = pd.read_sql(
    """
    SELECT
        product_id,
        COUNT(*) AS row_count
    FROM silver.product_categories
    GROUP BY product_id
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    """,
    engine
)

display(product_category_duplicates.head(20))

,product_id,row_count
0,PROD-00052,2
1,PROD-00065,2
2,PROD-00078,2
3,PROD-00039,2
4,PROD-00013,2
5,PROD-00026,2


In [14]:
pd.read_sql(
    """
    SELECT
        product_id,
        category_id,
        category_name,
        valid_from_utc,
        valid_to_utc
    FROM silver.product_categories
    WHERE product_id = 'PROD-00052'
    ORDER BY valid_from_utc
    """,
    engine
)

,product_id,category_id,category_name,valid_from_utc,valid_to_utc
0,PROD-00052,CAT-03,fashion,2026-06-22T00:00:00Z,2026-08-10T23:59:59Z
1,PROD-00052,CAT-02,home,2026-08-11T00:00:00Z,2026-09-21T00:00:00Z


In [15]:
product_category_test = pd.read_sql(
    """
    SELECT
        oi.order_id,
        oi.product_id,
        o.ordered_at_utc AS order_date,
        pc.category_name
    FROM silver.order_items oi
    JOIN silver.orders o
        ON oi.order_id = o.order_id
    LEFT JOIN silver.product_categories pc
        ON oi.product_id = pc.product_id
       AND o.ordered_at_utc >= pc.valid_from_utc
       AND o.ordered_at_utc <= pc.valid_to_utc
    WHERE oi.product_id = 'PROD-00052'
    ORDER BY o.ordered_at_utc
    LIMIT 20
    """,
    engine
)

display(product_category_test)

,order_id,product_id,order_date,category_name
0,ORD-006556,PROD-00052,2026-06-22T06:32:00Z,fashion
1,ORD-008970,PROD-00052,2026-06-22T11:04:00Z,fashion
2,ORD-008777,PROD-00052,2026-06-22T15:53:00Z,fashion
3,ORD-001984,PROD-00052,2026-06-22T21:28:00Z,fashion
4,ORD-000636,PROD-00052,2026-06-23T05:00:00Z,fashion
5,ORD-002643,PROD-00052,2026-06-23T08:33:00Z,fashion
6,ORD-006810,PROD-00052,2026-06-23T08:53:00Z,fashion
7,ORD-007239,PROD-00052,2026-06-23T14:02:00Z,fashion
8,ORD-007528,PROD-00052,2026-06-24T19:19:00Z,fashion
9,ORD-009797,PROD-00052,2026-06-24T21:21:00Z,fashion


In [16]:
product_category_duplicates = pd.read_sql(
    """
    SELECT
        oi.order_id,
        oi.product_id,
        COUNT(*) AS category_matches
    FROM silver.order_items oi
    JOIN silver.orders o
        ON oi.order_id = o.order_id
    LEFT JOIN silver.product_categories pc
        ON oi.product_id = pc.product_id
       AND o.ordered_at_utc >= pc.valid_from_utc
       AND o.ordered_at_utc <= pc.valid_to_utc
    GROUP BY
        oi.order_id,
        oi.product_id
    HAVING COUNT(*) > 1
    ORDER BY category_matches DESC
    """,
    engine
)

display(product_category_duplicates.head(20))

,order_id,product_id,category_matches
0,ORD-000559,PROD-00050,3
1,ORD-003391,PROD-00011,3
2,ORD-000011,PROD-MISSING,3
3,ORD-001184,PROD-00069,2
4,ORD-008271,PROD-00025,2
5,ORD-001301,PROD-00050,2
6,ORD-007261,PROD-00030,2
7,ORD-009171,PROD-00023,2
8,ORD-006386,PROD-00029,2
9,ORD-001093,PROD-00058,2


In [17]:
pd.read_sql(
    """
    SELECT
        oi.order_id,
        oi.product_id,
        oi.order_item_id,
        o.ordered_at_utc AS order_date,
        pc.category_id,
        pc.category_name,
        pc.valid_from_utc,
        pc.valid_to_utc
    FROM silver.order_items oi
    JOIN silver.orders o
        ON oi.order_id = o.order_id
    LEFT JOIN silver.product_categories pc
        ON oi.product_id = pc.product_id
       AND o.ordered_at_utc >= pc.valid_from_utc
       AND o.ordered_at_utc <= pc.valid_to_utc
    WHERE oi.order_id = 'ORD-000559'
      AND oi.product_id = 'PROD-00050'
    ORDER BY oi.order_item_id
    """,
    engine
)

,order_id,product_id,order_item_id,order_date,category_id,category_name,valid_from_utc,valid_to_utc
0,ORD-000559,PROD-00050,ITEM-000559-01,2026-08-08T02:35:00Z,CAT-04,grocery,2026-06-22T00:00:00Z,2026-09-21T00:00:00Z
1,ORD-000559,PROD-00050,ITEM-000559-03,2026-08-08T02:35:00Z,CAT-04,grocery,2026-06-22T00:00:00Z,2026-09-21T00:00:00Z
2,ORD-000559,PROD-00050,ITEM-000559-04,2026-08-08T02:35:00Z,CAT-04,grocery,2026-06-22T00:00:00Z,2026-09-21T00:00:00Z


In [18]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_orders,
        COUNT(store_id) AS orders_with_store,
        COUNT(*) - COUNT(store_id) AS orders_without_store
    FROM silver.orders
    """,
    engine
)

,total_orders,orders_with_store,orders_without_store
0,10001,2517,7484


In [19]:
pd.read_sql(
    """
    SELECT
        sales_channel,
        COUNT(*) AS total_orders,
        COUNT(store_id) AS orders_with_store,
        COUNT(*) - COUNT(store_id) AS orders_without_store
    FROM silver.orders
    GROUP BY sales_channel
    ORDER BY total_orders DESC
    """,
    engine
)

,sales_channel,total_orders,orders_with_store,orders_without_store
0,STORE,2517,2517,0
1,WEB,2507,0,2507
2,MOBILE_APP,2495,0,2495
3,MARKETPLACE,2482,0,2482


In [20]:
sales_channels_check = pd.read_sql(
    """
    SELECT *
    FROM silver.sales_channels
    ORDER BY 1
    """,
    engine
)

display(sales_channels_check)

,channel_id,channel_name,source_row_id,ingested_at_utc
0,MARKETPLACE,MARKETPLACE,CHANNEL-ROW-03,2026-09-22 20:52:05.473308+00:00
1,MOBILE_APP,MOBILE_APP,CHANNEL-ROW-02,2026-09-22 20:52:05.473308+00:00
2,STORE,STORE,CHANNEL-ROW-04,2026-09-22 20:52:05.473308+00:00
3,WEB,WEB,CHANNEL-ROW-01,2026-09-22 20:52:05.473308+00:00


In [21]:
pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_rows,
        COUNT(DISTINCT order_id) AS unique_orders
    FROM silver.orders
    """,
    engine
)

,total_rows,unique_orders
0,10001,10000


In [23]:
orders_dedup = pd.read_sql(
    """
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY updated_at_utc DESC,
                         ingested_at_utc DESC,
                         source_row_id
            ) AS rn
        FROM silver.orders
    ) x
    WHERE rn = 1
    """,
    engine
)

print("Rows setelah dedup :", len(orders_dedup))
print("Unique order_id    :", orders_dedup["order_id"].nunique())

Rows setelah dedup : 10000
Unique order_id    : 10000


In [24]:
print("Rows setelah dedup:", len(orders_dedup))
print("Unique order_id:", orders_dedup["order_id"].nunique())

Rows setelah dedup: 10000
Unique order_id: 10000


In [25]:
order_items_summary = pd.read_sql(
    """
    SELECT
        order_id,
        COUNT(order_item_id) AS item_count,
        SUM(quantity) AS unit_quantity,
        SUM(quantity * unit_price) AS gross_merchandise_value,
        SUM(item_discount_amount) AS discount_amount
    FROM silver.order_items
    GROUP BY order_id
    ORDER BY order_id
    LIMIT 10
    """,
    engine
)

display(order_items_summary)

,order_id,item_count,unit_quantity,gross_merchandise_value,discount_amount
0,ORD-000001,2,4.0,1118.24,0.0
1,ORD-000002,2,5.0,1503.35,0.0
2,ORD-000003,1,3.0,278.49,5.0
3,ORD-000004,3,11.0,2396.58,5.0
4,ORD-000005,4,8.0,1883.20,5.0
5,ORD-000006,4,11.0,2200.22,0.0
6,ORD-000007,4,7.0,2131.88,10.0
7,ORD-000008,3,7.0,980.93,5.0
8,ORD-000009,4,9.0,2812.32,0.0
9,ORD-000010,1,4.0,1503.56,0.0


In [26]:
payment_summary = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        SUM("payload.amount") AS captured_payment_amount
    FROM silver.payment_events
    WHERE event_type = 'PAYMENT_CAPTURED'
    GROUP BY "payload.order_id"
    ORDER BY "payload.order_id"
    LIMIT 10
    """,
    engine
)

display(payment_summary)

,order_id,captured_payment_amount
0,ORD-000001,1125.24
1,ORD-000002,1490.35
2,ORD-000003,270.99
3,ORD-000004,2361.58
4,ORD-000005,1867.70
5,ORD-000007,2482.04
6,ORD-000008,955.93
7,ORD-000009,2800.82
8,ORD-000010,1473.56
9,ORD-000011,2619.25


In [27]:
payment_status_summary = pd.read_sql(
    """
    SELECT
        order_id,
        event_type AS payment_status,
        occurred_at_utc AS latest_payment_event_at
    FROM (
        SELECT
            "payload.order_id" AS order_id,
            event_type,
            occurred_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY "payload.order_id"
                ORDER BY occurred_at_utc DESC, event_id DESC
            ) AS rn
        FROM silver.payment_events
    ) x
    WHERE rn = 1
    ORDER BY order_id
    LIMIT 20
    """,
    engine
)

display(payment_status_summary)

,order_id,payment_status,latest_payment_event_at
0,ORD-000001,PAYMENT_CAPTURED,2026-08-20T07:16:00Z
1,ORD-000002,PAYMENT_CAPTURED,2026-08-16T07:38:00Z
2,ORD-000003,PAYMENT_AUTHORIZED,2026-07-05T00:08:00+07:00
3,ORD-000004,PAYMENT_CAPTURED,2026-08-28T14:09:00Z
4,ORD-000005,PAYMENT_AUTHORIZED,2026-07-06T22:07:00+07:00
5,ORD-000006,PAYMENT_FAILED,2026-08-08T19:59:00Z
6,ORD-000007,PAYMENT_CAPTURED,2026-07-18T13:24:00+07:00
7,ORD-000008,PAYMENT_CAPTURED,2026-08-28T21:41:00Z
8,ORD-000009,PAYMENT_CAPTURED,2026-07-15T13:02:00Z
9,ORD-000010,PAYMENT_CAPTURED,2026-07-30T03:08:00Z


In [28]:
refund_summary = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        SUM("payload.amount") AS refunded_amount
    FROM silver.refund_events
    WHERE event_type = 'REFUND_COMPLETED'
    GROUP BY "payload.order_id"
    ORDER BY "payload.order_id"
    LIMIT 10
    """,
    engine
)

display(refund_summary)

,order_id,refunded_amount
0,ORD-000018,1063.59
1,ORD-000035,432.69
2,ORD-000038,275.21
3,ORD-000039,141.39
4,ORD-000044,340.56
5,ORD-000052,1260.65
6,ORD-000053,288.59
7,ORD-000058,3203.06
8,ORD-000066,299.07
9,ORD-000069,1072.71


In [29]:
return_summary = pd.read_sql(
    """
    SELECT
        order_id,
        event_type AS return_status,
        occurred_at_utc AS return_status_at
    FROM (
        SELECT
            "payload.order_id" AS order_id,
            event_type,
            occurred_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY "payload.order_id"
                ORDER BY occurred_at_utc DESC, event_id DESC
            ) AS rn
        FROM silver.return_events
    ) x
    WHERE rn = 1
    ORDER BY order_id
    LIMIT 20
    """,
    engine
)

display(return_summary)

,order_id,return_status,return_status_at
0,ORD-000003,RETURN_REQUESTED,2026-07-09T17:03:00Z
1,ORD-000011,RETURN_CLOSED,2026-07-04T14:37:00Z
2,ORD-000016,RETURN_RECEIVED,2026-07-17T13:26:00Z
3,ORD-000029,RETURN_CLOSED,2026-08-06T18:32:00Z
4,ORD-000034,RETURN_RECEIVED,2026-09-02T17:54:00Z
5,ORD-000039,RETURN_CLOSED,2026-07-24T14:14:00Z
6,ORD-000043,RETURN_CLOSED,2026-09-04T22:43:00Z
7,ORD-000044,RETURN_CLOSED,2026-08-21T22:02:00Z
8,ORD-000049,RETURN_REQUESTED,2026-08-29T09:12:00Z
9,ORD-000056,RETURN_REQUESTED,2026-08-26T10:38:00Z


In [30]:
promotion_summary = pd.read_sql(
    """
    SELECT
        order_id,
        COUNT(DISTINCT promotion_id) AS promotion_count
    FROM silver.order_promotions
    GROUP BY order_id
    ORDER BY order_id
    LIMIT 10
    """,
    engine
)

display(promotion_summary)

,order_id,promotion_count
0,ORD-000001,1
1,ORD-000002,2
2,ORD-000003,1
3,ORD-000004,2
4,ORD-000005,2
5,ORD-000006,1
6,ORD-000007,2
7,ORD-000008,2
8,ORD-000009,1
9,ORD-000010,2


In [31]:
first_order_summary = pd.read_sql(
    """
    SELECT
        order_id,
        CASE
            WHEN ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY ordered_at_utc, order_id
            ) = 1
            THEN TRUE
            ELSE FALSE
        END AS first_order_flag
    FROM (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    ) o
    ORDER BY order_id
    LIMIT 20
    """,
    engine
)

display(first_order_summary)

,order_id,first_order_flag
0,ORD-000001,False
1,ORD-000002,False
2,ORD-000003,True
3,ORD-000004,False
4,ORD-000005,False
5,ORD-000006,False
6,ORD-000007,False
7,ORD-000008,False
8,ORD-000009,False
9,ORD-000010,False


In [32]:
first_order_check = pd.read_sql(
    """
    SELECT
        COUNT(*) AS total_orders,
        COUNT(DISTINCT customer_id) AS unique_customers,
        COUNT(*) FILTER (
            WHERE rn = 1
        ) AS first_orders
    FROM (
        SELECT
            order_id,
            customer_id,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY ordered_at_utc, order_id
            ) AS rn
        FROM (
            SELECT *
            FROM (
                SELECT
                    *,
                    ROW_NUMBER() OVER (
                        PARTITION BY order_id
                        ORDER BY updated_at_utc DESC,
                                 ingested_at_utc DESC,
                                 source_row_id
                    ) AS order_rn
                FROM silver.orders
            ) x
            WHERE order_rn = 1
        ) o
    ) x
    """,
    engine
)

display(first_order_check)

,total_orders,unique_customers,first_orders
0,10000,2455,2455


In [33]:
stores_check = pd.read_sql(
    """
    SELECT *
    FROM silver.stores
    ORDER BY 1
    LIMIT 20
    """,
    engine
)

display(stores_check)

,city_id,store_id,store_name,location_type,source_row_id,ingested_at_utc
0,BDG,STORE-01,Retail Store 01,physical,STORE-ROW-01,2026-09-22 20:52:05.445811+00:00
1,DPS,STORE-04,Retail Store 04,physical,STORE-ROW-04,2026-09-22 20:52:05.445811+00:00
2,MDN,STORE-03,Retail Store 03,physical,STORE-ROW-03,2026-09-22 20:52:05.445811+00:00
3,MKS,STORE-05,Retail Store 05,physical,STORE-ROW-05,2026-09-22 20:52:05.445811+00:00
4,SBY,STORE-02,Retail Store 02,physical,STORE-ROW-02,2026-09-22 20:52:05.445811+00:00


In [5]:
order_360_test = pd.read_sql(
    """
    WITH orders_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    ),

    order_items_summary AS (
        SELECT
            order_id,
            COUNT(order_item_id) AS item_count,
            SUM(quantity) AS unit_quantity,
            SUM(quantity * unit_price) AS gross_merchandise_value,
            SUM(item_discount_amount) AS discount_amount
        FROM silver.order_items
        GROUP BY order_id
    ),

    payment_summary AS (
        SELECT
            "payload.order_id" AS order_id,
            SUM("payload.amount") AS captured_payment_amount
        FROM silver.payment_events
        WHERE event_type = 'PAYMENT_CAPTURED'
        GROUP BY "payload.order_id"
    ),

    payment_status_summary AS (
        SELECT
            order_id,
            event_type AS payment_status
        FROM (
            SELECT
                "payload.order_id" AS order_id,
                event_type,
                occurred_at_utc,
                event_id,
                ROW_NUMBER() OVER (
                    PARTITION BY "payload.order_id"
                    ORDER BY occurred_at_utc DESC, event_id DESC
                ) AS rn
            FROM silver.payment_events
        ) x
        WHERE rn = 1
    ),

    refund_summary AS (
        SELECT
            "payload.order_id" AS order_id,
            SUM("payload.amount") AS refunded_amount
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
        GROUP BY "payload.order_id"
    ),

    return_summary AS (
        SELECT
            order_id,
            event_type AS return_status
        FROM (
            SELECT
                "payload.order_id" AS order_id,
                event_type,
                occurred_at_utc,
                event_id,
                ROW_NUMBER() OVER (
                    PARTITION BY "payload.order_id"
                    ORDER BY occurred_at_utc DESC, event_id DESC
                ) AS rn
            FROM silver.return_events
        ) x
        WHERE rn = 1
    ),

    promotion_summary AS (
        SELECT
            order_id,
            COUNT(DISTINCT promotion_id) AS promotion_count
        FROM silver.order_promotions
        GROUP BY order_id
    ),

    first_order_summary AS (
        SELECT
            order_id,
            CASE
                WHEN ROW_NUMBER() OVER (
                    PARTITION BY customer_id
                    ORDER BY ordered_at_utc, order_id
                ) = 1
                THEN TRUE
                ELSE FALSE
            END AS first_order_flag
        FROM orders_dedup
    )

    SELECT
        o.order_id,
        o.customer_id,
        o.ordered_at_utc::DATE AS order_date,
        o.sales_channel,
        o.store_id,

        COALESCE(oi.gross_merchandise_value, 0) AS gross_merchandise_value,
        COALESCE(oi.discount_amount, 0) AS discount_amount,
        COALESCE(o.shipping_revenue, 0) AS shipping_revenue,

        COALESCE(ps.captured_payment_amount, 0) AS captured_payment_amount,
        COALESCE(rf.refunded_amount, 0) AS refunded_amount,

        (
            COALESCE(oi.gross_merchandise_value, 0)
            - COALESCE(oi.discount_amount, 0)
            + COALESCE(o.shipping_revenue, 0)
            - COALESCE(rf.refunded_amount, 0)
        ) AS net_revenue,

        COALESCE(oi.item_count, 0) AS item_count,
        COALESCE(oi.unit_quantity, 0) AS unit_quantity,

        o.status AS order_status,

        COALESCE(pst.payment_status, 'NO_PAYMENT') AS payment_status,
        COALESCE(rs.return_status, 'NO_RETURN') AS return_status,

        COALESCE(pr.promotion_count, 0) AS promotion_count,

        fo.first_order_flag

    FROM orders_dedup o

    LEFT JOIN order_items_summary oi
        ON o.order_id = oi.order_id

    LEFT JOIN payment_summary ps
        ON o.order_id = ps.order_id

    LEFT JOIN payment_status_summary pst
        ON o.order_id = pst.order_id

    LEFT JOIN refund_summary rf
        ON o.order_id = rf.order_id

    LEFT JOIN return_summary rs
        ON o.order_id = rs.order_id

    LEFT JOIN promotion_summary pr
        ON o.order_id = pr.order_id

    LEFT JOIN first_order_summary fo
        ON o.order_id = fo.order_id

    ORDER BY o.order_id
    """,
    engine
)

display(order_360_test.head(20))

,order_id,customer_id,order_date,sales_channel,store_id,gross_merchandise_value,discount_amount,shipping_revenue,captured_payment_amount,refunded_amount,net_revenue,item_count,unit_quantity,order_status,payment_status,return_status,promotion_count,first_order_flag
0,ORD-000001,CUST-01757,2026-08-20,MOBILE_APP,None,1118.24,0.0,12.0,1125.24,0.00,1130.24,2,4.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,1,False
1,ORD-000002,CUST-01160,2026-08-16,MOBILE_APP,None,1503.35,0.0,0.0,1490.35,0.00,1503.35,2,5.0,RETURNED,PAYMENT_CAPTURED,NO_RETURN,2,False
2,ORD-000003,CUST-00962,2026-07-04,WEB,None,278.49,5.0,7.5,270.99,0.00,280.99,1,3.0,PLACED,PAYMENT_AUTHORIZED,RETURN_REQUESTED,1,True
3,ORD-000004,CUST-00181,2026-08-28,MARKETPLACE,None,2396.58,5.0,0.0,2361.58,0.00,2391.58,3,11.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False
4,ORD-000005,CUST-02298,2026-07-06,STORE,STORE-03,1883.20,5.0,7.5,1867.70,0.00,1885.70,4,8.0,RETURNED,PAYMENT_AUTHORIZED,NO_RETURN,2,False
5,ORD-000006,CUST-00630,2026-08-08,WEB,None,2200.22,0.0,0.0,0.00,0.00,2200.22,4,11.0,CANCELLED,PAYMENT_FAILED,NO_RETURN,1,False
6,ORD-000007,CUST-00632,2026-07-18,STORE,STORE-03,2131.88,10.0,0.0,2482.04,0.00,2121.88,4,7.0,FULFILLED,PAYMENT_CAPTURED,NO_RETURN,2,False
7,ORD-000008,CUST-01792,2026-08-28,WEB,None,980.93,5.0,0.0,955.93,0.00,975.93,3,7.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False
8,ORD-000009,CUST-01220,2026-07-15,WEB,None,2812.32,0.0,3.5,2800.82,0.00,2815.82,4,9.0,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN,1,False
9,ORD-000010,CUST-00459,2026-07-30,MOBILE_APP,None,1503.56,0.0,0.0,1473.56,0.00,1503.56,1,4.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False


In [35]:
order_360_check = pd.DataFrame({
    "metric": [
        "total_rows",
        "unique_order_id",
        "duplicate_rows"
    ],
    "value": [
        len(order_360_test),
        order_360_test["order_id"].nunique(),
        len(order_360_test) - order_360_test["order_id"].nunique()
    ]
})

display(order_360_check)

,metric,value
0,total_rows,10000
1,unique_order_id,10000
2,duplicate_rows,0


In [36]:
null_check = order_360_test.isna().sum()

display(
    null_check[null_check > 0]
)

store_id    7483
dtype: int64

In [37]:
net_revenue_check = order_360_test[
    order_360_test["net_revenue"].round(2)
    != (
        order_360_test["gross_merchandise_value"]
        - order_360_test["discount_amount"]
        + order_360_test["shipping_revenue"]
        - order_360_test["refunded_amount"]
    ).round(2)
]

print("Jumlah order dengan net_revenue tidak sesuai:", len(net_revenue_check))

display(net_revenue_check.head(20))

Jumlah order dengan net_revenue tidak sesuai: 0


,order_id,customer_id,order_date,sales_channel,store_id,gross_merchandise_value,discount_amount,shipping_revenue,captured_payment_amount,refunded_amount,net_revenue,item_count,unit_quantity,order_status,payment_status,return_status,promotion_count,first_order_flag


In [39]:
payment_check = order_360_test[
    (order_360_test["captured_payment_amount"] < 0)
    |
    (
        (order_360_test["payment_status"] == "PAYMENT_CAPTURED")
        & (order_360_test["captured_payment_amount"] <= 0)
    )
]

print(
    "Jumlah order dengan captured payment tidak valid:",
    len(payment_check)
)

display(payment_check.head(20))

Jumlah order dengan captured payment tidak valid: 8


,order_id,customer_id,order_date,sales_channel,store_id,gross_merchandise_value,discount_amount,shipping_revenue,captured_payment_amount,refunded_amount,net_revenue,item_count,unit_quantity,order_status,payment_status,return_status,promotion_count,first_order_flag
510,ORD-000511,CUST-00197,2026-09-06,MARKETPLACE,None,20.09,2.5,0.0,0.0,0.0,17.59,1,1.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False
2278,ORD-002279,CUST-01518,2026-08-06,MOBILE_APP,None,11.48,0.0,0.0,0.0,0.0,11.48,1,1.0,PLACED,PAYMENT_CAPTURED,RETURN_CLOSED,2,False
2405,ORD-002406,CUST-02474,2026-07-13,WEB,None,17.31,2.5,0.0,0.0,0.0,14.81,1,1.0,FULFILLED,PAYMENT_CAPTURED,NO_RETURN,2,True
3651,ORD-003652,CUST-00226,2026-08-11,MOBILE_APP,None,11.48,0.0,0.0,0.0,0.0,11.48,1,1.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False
5052,ORD-005053,CUST-00252,2026-07-21,MOBILE_APP,None,17.31,2.5,0.0,0.0,0.0,14.81,1,1.0,RETURNED,PAYMENT_CAPTURED,NO_RETURN,2,False
7665,ORD-007666,CUST-02035,2026-09-14,MARKETPLACE,None,11.48,0.0,3.5,0.0,0.0,14.98,1,1.0,PLACED,PAYMENT_CAPTURED,NO_RETURN,2,False
7705,ORD-007706,CUST-00445,2026-06-26,WEB,None,22.96,0.0,0.0,0.0,0.0,22.96,1,2.0,FULFILLED,PAYMENT_CAPTURED,NO_RETURN,2,True
8158,ORD-008159,CUST-00803,2026-06-29,STORE,STORE-04,11.48,0.0,0.0,0.0,0.0,11.48,1,1.0,FULFILLED,PAYMENT_CAPTURED,NO_RETURN,2,True


In [40]:
display(
    order_360_test
    .groupby("payment_status")["captured_payment_amount"]
    .agg(["count", "min", "max", "mean"])
)

,count,min,max,mean
payment_status,,,,
PAYMENT_AUTHORIZED,881,0.0,5340.64,1274.306209
PAYMENT_CAPTURED,8013,0.0,8493.14,1549.069354
PAYMENT_FAILED,1106,0.0,0.00,0.000000


In [41]:
payment_invalid_orders = order_360_test[
    (order_360_test["payment_status"] == "PAYMENT_CAPTURED")
    & (order_360_test["captured_payment_amount"] == 0)
]

invalid_order_ids = payment_invalid_orders["order_id"].tolist()

payment_invalid_events = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        "payload.payment_id" AS payment_id,
        event_type,
        "payload.amount" AS amount,
        occurred_at_utc
    FROM silver.payment_events
    WHERE "payload.order_id" = ANY(%s)
    ORDER BY "payload.order_id", occurred_at_utc
    """,
    engine,
    params=(invalid_order_ids,)
)

display(payment_invalid_events)

,order_id,payment_id,event_type,amount,occurred_at_utc
0,ORD-000511,PAY-ORD-000511,PAYMENT_AUTHORIZED,0.0,2026-09-06T04:31:00Z
1,ORD-000511,PAY-ORD-000511,PAYMENT_CAPTURED,0.0,2026-09-06T05:26:00Z
2,ORD-002279,PAY-ORD-002279,PAYMENT_AUTHORIZED,0.0,2026-08-06T18:36:00Z
3,ORD-002279,PAY-ORD-002279,PAYMENT_CAPTURED,0.0,2026-08-06T19:31:00Z
4,ORD-002406,PAY-ORD-002406,PAYMENT_AUTHORIZED,0.0,2026-07-13T12:56:00Z
5,ORD-002406,PAY-ORD-002406,PAYMENT_CAPTURED,0.0,2026-07-13T13:51:00Z
6,ORD-003652,PAY-ORD-003652,PAYMENT_AUTHORIZED,0.0,2026-08-11T15:04:00Z
7,ORD-003652,PAY-ORD-003652,PAYMENT_CAPTURED,0.0,2026-08-11T15:59:00Z
8,ORD-005053,PAY-ORD-005053,PAYMENT_AUTHORIZED,0.0,2026-07-21T04:54:00Z
9,ORD-005053,PAY-ORD-005053,PAYMENT_CAPTURED,0.0,2026-07-21T05:49:00Z


In [42]:
refund_check = order_360_test[
    order_360_test["refunded_amount"] < 0
]

print(
    "Jumlah order dengan refunded_amount negatif:",
    len(refund_check)
)

display(refund_check.head(20))

Jumlah order dengan refunded_amount negatif: 0


,order_id,customer_id,order_date,sales_channel,store_id,gross_merchandise_value,discount_amount,shipping_revenue,captured_payment_amount,refunded_amount,net_revenue,item_count,unit_quantity,order_status,payment_status,return_status,promotion_count,first_order_flag


In [43]:
refund_total_check = pd.read_sql(
    """
    SELECT
        SUM("payload.amount") AS silver_refund_total
    FROM silver.refund_events
    WHERE event_type = 'REFUND_COMPLETED'
    """,
    engine
)

silver_refund_total = refund_total_check.iloc[0]["silver_refund_total"]
gold_refund_total = order_360_test["refunded_amount"].sum()

print("Silver refund total:", silver_refund_total)
print("Gold refund total:", gold_refund_total)
print(
    "Selisih:",
    round(float(silver_refund_total) - float(gold_refund_total), 2)
)

Silver refund total: 1606522.9900000002
Gold refund total: 1606522.9900000002
Selisih: 0.0


In [44]:
return_status_check = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        COUNT(*) AS return_event_count,
        STRING_AGG(event_type, ', ' ORDER BY occurred_at_utc) AS return_events
    FROM silver.return_events
    GROUP BY "payload.order_id"
    ORDER BY "payload.order_id"
    """,
    engine
)

display(return_status_check.head(20))

,order_id,return_event_count,return_events
0,ORD-000003,1,RETURN_REQUESTED
1,ORD-000011,3,"RETURN_REQUESTED, RETURN_RECEIVED, RETURN_CLOSED"
2,ORD-000016,2,"RETURN_REQUESTED, RETURN_RECEIVED"
3,ORD-000029,3,"RETURN_REQUESTED, RETURN_RECEIVED, RETURN_CLOSED"
4,ORD-000034,3,"RETURN_REQUESTED, RETURN_RECEIVED, RETURN_RECE..."
5,ORD-000039,3,"RETURN_REQUESTED, RETURN_RECEIVED, RETURN_CLOSED"
6,ORD-000043,3,"RETURN_REQUESTED, RETURN_RECEIVED, RETURN_CLOSED"
7,ORD-000044,3,"RETURN_REQUESTED, RETURN_RECEIVED, RETURN_CLOSED"
8,ORD-000049,1,RETURN_REQUESTED
9,ORD-000056,1,RETURN_REQUESTED


In [45]:
return_compare = pd.read_sql(
    """
    WITH ranked_returns AS (
        SELECT
            "payload.order_id" AS order_id,
            event_type,
            occurred_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY "payload.order_id"
                ORDER BY occurred_at_utc DESC, event_id DESC
            ) AS rn
        FROM silver.return_events
    )
    SELECT
        order_id,
        event_type AS silver_return_status,
        occurred_at_utc
    FROM ranked_returns
    WHERE rn = 1
    """,
    engine
)

return_compare = return_compare.merge(
    order_360_test[["order_id", "return_status"]],
    on="order_id",
    how="left"
)

return_compare["return_status"] = return_compare["return_status"].fillna("NO_RETURN")

return_mismatch = return_compare[
    return_compare["silver_return_status"]
    != return_compare["return_status"]
]

print("Jumlah return status mismatch:", len(return_mismatch))

display(return_mismatch.head(20))

Jumlah return status mismatch: 0


,order_id,silver_return_status,occurred_at_utc,return_status


In [46]:
first_order_compare = pd.read_sql(
    """
    WITH dedup_orders AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) x
        WHERE rn = 1
    ),
    ranked_orders AS (
        SELECT
            order_id,
            customer_id,
            ROW_NUMBER() OVER (
                PARTITION BY customer_id
                ORDER BY ordered_at_utc, order_id
            ) AS customer_order_rank
        FROM dedup_orders
    )
    SELECT
        order_id,
        customer_id,
        (customer_order_rank = 1) AS expected_first_order_flag
    FROM ranked_orders
    """,
    engine
)

first_order_compare = first_order_compare.merge(
    order_360_test[["order_id", "first_order_flag"]],
    on="order_id",
    how="left"
)

first_order_mismatch = first_order_compare[
    first_order_compare["expected_first_order_flag"]
    != first_order_compare["first_order_flag"]
]

print(
    "Jumlah first_order_flag mismatch:",
    len(first_order_mismatch)
)

display(first_order_mismatch.head(20))

Jumlah first_order_flag mismatch: 0


,order_id,customer_id,expected_first_order_flag,first_order_flag


In [6]:
expected_columns = [
    "order_id",
    "customer_id",
    "order_date",
    "sales_channel",
    "store_id",
    "gross_merchandise_value",
    "discount_amount",
    "shipping_revenue",
    "captured_payment_amount",
    "refunded_amount",
    "net_revenue",
    "item_count",
    "unit_quantity",
    "order_status",
    "payment_status",
    "return_status",
    "promotion_count",
    "first_order_flag"
]

print("Jumlah kolom aktual:", len(order_360_test.columns))
print("Jumlah kolom expected:", len(expected_columns))

print("\nKolom yang belum ada:")
print([c for c in expected_columns if c not in order_360_test.columns])

print("\nKolom tambahan:")
print([c for c in order_360_test.columns if c not in expected_columns])

Jumlah kolom aktual: 18
Jumlah kolom expected: 18

Kolom yang belum ada:
[]

Kolom tambahan:
[]


In [7]:
required_columns = [
    "order_id",
    "customer_id",
    "order_date",
    "sales_channel",
    "gross_merchandise_value",
    "discount_amount",
    "shipping_revenue",
    "captured_payment_amount",
    "refunded_amount",
    "net_revenue",
    "item_count",
    "unit_quantity",
    "order_status",
    "payment_status",
    "return_status",
    "promotion_count",
    "first_order_flag"
]

null_check = order_360_test[required_columns].isna().sum()

print("NULL pada kolom NOT NULL:")
display(null_check[null_check > 0])

NULL pada kolom NOT NULL:


Series([], dtype: int64)

In [8]:
print("Data types:")
display(order_360_test.dtypes)

print("\nJumlah baris:", len(order_360_test))
print("Unique order_id:", order_360_test["order_id"].nunique())

Data types:


order_id                    object
customer_id                 object
order_date                  object
sales_channel               object
store_id                    object
gross_merchandise_value    float64
discount_amount            float64
shipping_revenue           float64
captured_payment_amount    float64
refunded_amount            float64
net_revenue                float64
item_count                   int64
unit_quantity              float64
order_status                object
payment_status              object
return_status               object
promotion_count              int64
first_order_flag              bool
dtype: object


Jumlah baris: 10000
Unique order_id: 10000


In [9]:
numeric_columns = [
    "gross_merchandise_value",
    "discount_amount",
    "shipping_revenue",
    "captured_payment_amount",
    "refunded_amount",
    "net_revenue",
    "item_count",
    "unit_quantity",
    "promotion_count"
]

display(order_360_test[numeric_columns].describe().T)

,count,mean,std,min,25%,50%,75%,max
gross_merchandise_value,10000.0,1510.244163,973.043645,11.48,722.78,1386.890,2144.1250,5820.72
discount_amount,10000.0,3.717750,3.599688,0.00,0.00,2.500,5.0000,20.00
shipping_revenue,10000.0,4.489750,4.569219,0.00,0.00,3.500,7.5000,12.00
captured_payment_amount,10000.0,1353.535650,1094.687443,0.00,421.57,1210.025,2047.1800,8493.14
refunded_amount,10000.0,160.652299,472.301491,0.00,0.00,0.000,0.0000,6490.12
net_revenue,10000.0,1350.363864,991.608715,-3227.06,556.60,1203.950,1989.9275,5818.22
item_count,10000.0,2.496000,1.121566,1.00,1.00,2.000,4.0000,4.00
unit_quantity,10000.0,6.238500,3.319988,1.00,4.00,6.000,9.0000,16.00
promotion_count,10000.0,1.572900,0.657213,0.00,1.00,2.000,2.0000,2.00


In [10]:
negative_net_revenue = order_360_test[
    order_360_test["net_revenue"] < 0
].copy()

print("Jumlah order dengan net_revenue negatif:", len(negative_net_revenue))

display(
    negative_net_revenue[
        [
            "order_id",
            "gross_merchandise_value",
            "discount_amount",
            "shipping_revenue",
            "captured_payment_amount",
            "refunded_amount",
            "net_revenue",
            "order_status",
            "payment_status",
            "return_status"
        ]
    ].sort_values("net_revenue").head(20)
)

Jumlah order dengan net_revenue negatif: 24


,order_id,gross_merchandise_value,discount_amount,shipping_revenue,captured_payment_amount,refunded_amount,net_revenue,order_status,payment_status,return_status
747,ORD-000748,3258.06,2.5,7.5,3245.06,6490.12,-3.227060e+03,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN
8448,ORD-008449,3206.40,0.0,0.0,3201.40,6402.80,-3.196400e+03,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN
4973,ORD-004974,2836.94,0.0,0.0,2836.94,5673.88,-2.836940e+03,PLACED,PAYMENT_CAPTURED,NO_RETURN
7136,ORD-007137,2229.80,5.0,0.0,2209.80,4419.60,-2.194800e+03,PLACED,PAYMENT_CAPTURED,RETURN_REQUESTED
7233,ORD-007234,2172.15,10.0,7.5,2159.65,4319.30,-2.149650e+03,RETURNED,PAYMENT_CAPTURED,NO_RETURN
4264,ORD-004265,2126.42,15.0,0.0,2108.42,4216.84,-2.105420e+03,PLACED,PAYMENT_CAPTURED,NO_RETURN
6583,ORD-006584,2041.72,7.5,7.5,2031.72,4063.44,-2.021720e+03,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN
6430,ORD-006431,1797.91,2.5,3.5,0.00,3577.82,-1.778910e+03,CANCELLED,PAYMENT_AUTHORIZED,NO_RETURN
7020,ORD-007021,1606.36,0.0,7.5,1603.86,3207.72,-1.593860e+03,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN
5343,ORD-005344,1432.26,0.0,7.5,0.00,2859.52,-1.419760e+03,CANCELLED,PAYMENT_FAILED,NO_RETURN


In [11]:
negative_net_revenue["revenue_before_refund"] = (
    negative_net_revenue["gross_merchandise_value"]
    - negative_net_revenue["discount_amount"]
    + negative_net_revenue["shipping_revenue"]
).round(2)

negative_net_revenue["refund_exceeds_revenue"] = (
    negative_net_revenue["refunded_amount"]
    > negative_net_revenue["revenue_before_refund"]
)

display(
    negative_net_revenue[
        [
            "order_id",
            "gross_merchandise_value",
            "discount_amount",
            "shipping_revenue",
            "revenue_before_refund",
            "refunded_amount",
            "net_revenue",
            "order_status",
            "payment_status",
            "return_status",
        ]
    ].sort_values("net_revenue")
)

,order_id,gross_merchandise_value,discount_amount,shipping_revenue,revenue_before_refund,refunded_amount,net_revenue,order_status,payment_status,return_status
747,ORD-000748,3258.06,2.5,7.5,3263.06,6490.12,-3.227060e+03,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN
8448,ORD-008449,3206.40,0.0,0.0,3206.40,6402.80,-3.196400e+03,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN
4973,ORD-004974,2836.94,0.0,0.0,2836.94,5673.88,-2.836940e+03,PLACED,PAYMENT_CAPTURED,NO_RETURN
7136,ORD-007137,2229.80,5.0,0.0,2224.80,4419.60,-2.194800e+03,PLACED,PAYMENT_CAPTURED,RETURN_REQUESTED
7233,ORD-007234,2172.15,10.0,7.5,2169.65,4319.30,-2.149650e+03,RETURNED,PAYMENT_CAPTURED,NO_RETURN
4264,ORD-004265,2126.42,15.0,0.0,2111.42,4216.84,-2.105420e+03,PLACED,PAYMENT_CAPTURED,NO_RETURN
6583,ORD-006584,2041.72,7.5,7.5,2041.72,4063.44,-2.021720e+03,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN
6430,ORD-006431,1797.91,2.5,3.5,1798.91,3577.82,-1.778910e+03,CANCELLED,PAYMENT_AUTHORIZED,NO_RETURN
7020,ORD-007021,1606.36,0.0,7.5,1613.86,3207.72,-1.593860e+03,CONFIRMED,PAYMENT_CAPTURED,NO_RETURN
5343,ORD-005344,1432.26,0.0,7.5,1439.76,2859.52,-1.419760e+03,CANCELLED,PAYMENT_FAILED,NO_RETURN


In [12]:
print("Total negative net revenue:", len(negative_net_revenue))

print(
    "\nReturn status:"
)
display(
    negative_net_revenue["return_status"]
    .value_counts()
)

print(
    "\nRefund melebihi revenue sebelum refund:"
)
print(
    negative_net_revenue["refund_exceeds_revenue"].value_counts()
)

Total negative net revenue: 24

Return status:


return_status
NO_RETURN           19
RETURN_CLOSED        2
RETURN_REQUESTED     2
RETURN_RECEIVED      1
Name: count, dtype: int64


Refund melebihi revenue sebelum refund:
refund_exceeds_revenue
True     18
False     6
Name: count, dtype: int64


In [13]:
negative_order_ids = negative_net_revenue["order_id"].tolist()

refund_negative_check = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        COUNT(*) AS refund_event_count,
        SUM("payload.amount") AS silver_refunded_amount,
        STRING_AGG(event_type, ', ' ORDER BY occurred_at_utc) AS refund_events
    FROM silver.refund_events
    WHERE "payload.order_id" = ANY(%(order_ids)s)
    GROUP BY "payload.order_id"
    ORDER BY "payload.order_id"
    """,
    engine,
    params={"order_ids": negative_order_ids}
)

display(
    negative_net_revenue[
        [
            "order_id",
            "gross_merchandise_value",
            "discount_amount",
            "shipping_revenue",
            "refunded_amount",
            "net_revenue",
            "return_status"
        ]
    ].merge(
        refund_negative_check,
        on="order_id",
        how="left"
    ).sort_values("net_revenue")
)

,order_id,gross_merchandise_value,discount_amount,shipping_revenue,refunded_amount,net_revenue,return_status,refund_event_count,silver_refunded_amount,refund_events
0,ORD-000748,3258.06,2.5,7.5,6490.12,-3.227060e+03,NO_RETURN,3,9735.18,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
19,ORD-008449,3206.40,0.0,0.0,6402.80,-3.196400e+03,NO_RETURN,3,9604.20,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
6,ORD-004974,2836.94,0.0,0.0,5673.88,-2.836940e+03,NO_RETURN,3,8510.82,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
15,ORD-007137,2229.80,5.0,0.0,4419.60,-2.194800e+03,RETURN_REQUESTED,3,6629.40,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
16,ORD-007234,2172.15,10.0,7.5,4319.30,-2.149650e+03,NO_RETURN,3,6478.95,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
5,ORD-004265,2126.42,15.0,0.0,4216.84,-2.105420e+03,NO_RETURN,3,6325.26,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
13,ORD-006584,2041.72,7.5,7.5,4063.44,-2.021720e+03,NO_RETURN,3,6095.16,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
12,ORD-006431,1797.91,2.5,3.5,3577.82,-1.778910e+03,NO_RETURN,3,5366.73,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
14,ORD-007021,1606.36,0.0,7.5,3207.72,-1.593860e+03,NO_RETURN,3,4811.58,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"
8,ORD-005344,1432.26,0.0,7.5,2859.52,-1.419760e+03,NO_RETURN,3,4289.28,"REFUND_ISSUED, REFUND_COMPLETED, REFUND_COMPLETED"


In [14]:
print("Jumlah negative order:", len(negative_net_revenue))

print(
    "Negative order yang punya refund event:",
    negative_net_revenue["order_id"]
    .isin(refund_negative_check["order_id"])
    .sum()
)

print(
    "Negative order tanpa refund event:",
    (~negative_net_revenue["order_id"]
     .isin(refund_negative_check["order_id"]))
    .sum()
)

Jumlah negative order: 24
Negative order yang punya refund event: 24
Negative order tanpa refund event: 0


In [15]:
refund_amount_check = (
    negative_net_revenue[
        ["order_id", "refunded_amount"]
    ]
    .merge(
        refund_negative_check[
            ["order_id", "silver_refunded_amount"]
        ],
        on="order_id",
        how="left"
    )
)

refund_amount_check["difference"] = (
    refund_amount_check["refunded_amount"]
    - refund_amount_check["silver_refunded_amount"]
).round(2)

print(
    "Jumlah refund yang nominalnya berbeda:",
    (refund_amount_check["difference"] != 0).sum()
)

display(
    refund_amount_check[
        refund_amount_check["difference"] != 0
    ]
)

Jumlah refund yang nominalnya berbeda: 24


,order_id,refunded_amount,silver_refunded_amount,difference
0,ORD-000748,6490.12,9735.18,-3245.06
1,ORD-002412,1072.16,1608.24,-536.08
2,ORD-003361,2720.59,5441.18,-2720.59
3,ORD-003564,2182.02,3273.03,-1091.01
4,ORD-003596,2670.32,5340.64,-2670.32
5,ORD-004265,4216.84,6325.26,-2108.42
6,ORD-004974,5673.88,8510.82,-2836.94
7,ORD-005195,367.55,1102.65,-735.10
8,ORD-005344,2859.52,4289.28,-1429.76
9,ORD-005513,899.84,1349.76,-449.92


In [16]:
refund_detail_check = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        "payload.refund_id" AS refund_id,
        event_type,
        "payload.amount" AS amount,
        occurred_at_utc
    FROM silver.refund_events
    WHERE "payload.order_id" = ANY(%(order_ids)s)
      AND event_type = 'REFUND_COMPLETED'
    ORDER BY "payload.order_id", occurred_at_utc
    """,
    engine,
    params={"order_ids": negative_order_ids}
)

display(refund_detail_check)

,order_id,refund_id,event_type,amount,occurred_at_utc
0,ORD-000748,REF-ORD-000748-01,REFUND_COMPLETED,3245.06,2026-07-05T01:16:00Z
1,ORD-000748,REF-ORD-000748-01,REFUND_COMPLETED,3245.06,2026-07-05T01:16:00Z
2,ORD-002412,REF-ORD-002412-01,REFUND_COMPLETED,536.08,2026-09-16T13:36:00Z
3,ORD-002412,REF-ORD-002412-01,REFUND_COMPLETED,536.08,2026-09-16T13:36:00Z
4,ORD-003361,REF-ORD-003361-01,REFUND_COMPLETED,2720.59,2026-07-08T12:35:00Z
5,ORD-003564,REF-ORD-003564-01,REFUND_COMPLETED,1091.01,2026-06-30T01:24:00Z
6,ORD-003564,REF-ORD-003564-01,REFUND_COMPLETED,1091.01,2026-06-30T01:24:00Z
7,ORD-003596,REF-ORD-003596-01,REFUND_COMPLETED,2670.32,2026-07-22T12:55:00Z
8,ORD-004265,REF-ORD-004265-01,REFUND_COMPLETED,2108.42,2026-06-27T22:23:00Z
9,ORD-004265,REF-ORD-004265-01,REFUND_COMPLETED,2108.42,2026-06-27T22:23:00Z


In [17]:
refund_dedup_check = pd.read_sql(
    """
    WITH refund_dedup AS (
        SELECT
            "payload.order_id" AS order_id,
            "payload.refund_id" AS refund_id,
            MAX("payload.amount") AS refund_amount
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
          AND "payload.order_id" = ANY(%(order_ids)s)
        GROUP BY
            "payload.order_id",
            "payload.refund_id"
    )
    SELECT
        order_id,
        SUM(refund_amount) AS silver_refunded_amount
    FROM refund_dedup
    GROUP BY order_id
    ORDER BY order_id
    """,
    engine,
    params={"order_ids": negative_order_ids}
)

refund_amount_check = (
    negative_net_revenue[
        ["order_id", "refunded_amount"]
    ]
    .merge(
        refund_dedup_check,
        on="order_id",
        how="left"
    )
)

refund_amount_check["difference"] = (
    refund_amount_check["refunded_amount"]
    - refund_amount_check["silver_refunded_amount"]
).round(2)

print(
    "Jumlah refund yang nominalnya berbeda:",
    (refund_amount_check["difference"] != 0).sum()
)

display(
    refund_amount_check[
        refund_amount_check["difference"] != 0
    ]
)

Jumlah refund yang nominalnya berbeda: 18


,order_id,refunded_amount,silver_refunded_amount,difference
0,ORD-000748,6490.12,3245.06,3245.06
1,ORD-002412,1072.16,536.08,536.08
3,ORD-003564,2182.02,1091.01,1091.01
5,ORD-004265,4216.84,2108.42,2108.42
6,ORD-004974,5673.88,2836.94,2836.94
8,ORD-005344,2859.52,1429.76,1429.76
9,ORD-005513,899.84,449.92,449.92
12,ORD-006431,3577.82,1788.91,1788.91
13,ORD-006584,4063.44,2031.72,2031.72
14,ORD-007021,3207.72,1603.86,1603.86


In [18]:
refund_summary_correct = pd.read_sql(
    """
    WITH refund_dedup AS (
        SELECT
            "payload.order_id" AS order_id,
            "payload.refund_id" AS refund_id,
            MAX("payload.amount") AS refund_amount
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
        GROUP BY
            "payload.order_id",
            "payload.refund_id"
    )
    SELECT
        order_id,
        SUM(refund_amount) AS refunded_amount
    FROM refund_dedup
    GROUP BY order_id
    """,
    engine
)

display(refund_summary_correct.head(10))

,order_id,refunded_amount
0,ORD-001302,2607.19
1,ORD-003899,63.69
2,ORD-002601,776.99
3,ORD-009542,1235.86
4,ORD-006850,406.11
5,ORD-007047,195.53
6,ORD-001396,1297.86
7,ORD-004295,150.07
8,ORD-008977,1228.86
9,ORD-009953,1350.79


In [19]:
order_360_test = order_360_test.drop(
    columns=["refunded_amount"]
).merge(
    refund_summary_correct,
    on="order_id",
    how="left"
)

order_360_test["refunded_amount"] = (
    order_360_test["refunded_amount"]
    .fillna(0)
    .round(2)
)

In [20]:
order_360_test["net_revenue"] = (
    order_360_test["gross_merchandise_value"]
    - order_360_test["discount_amount"]
    + order_360_test["shipping_revenue"]
    - order_360_test["refunded_amount"]
).round(2)

In [21]:
silver_refund_total = pd.read_sql(
    """
    WITH refund_dedup AS (
        SELECT
            "payload.order_id" AS order_id,
            "payload.refund_id" AS refund_id,
            MAX("payload.amount") AS refund_amount
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
        GROUP BY
            "payload.order_id",
            "payload.refund_id"
    )
    SELECT SUM(refund_amount) AS total_refund
    FROM refund_dedup
    """,
    engine
).iloc[0]["total_refund"]

gold_test_refund_total = order_360_test["refunded_amount"].sum()

print("Silver refund total:", silver_refund_total)
print("Order 360 refund total:", gold_test_refund_total)
print(
    "Selisih:",
    round(float(silver_refund_total) - float(gold_test_refund_total), 2)
)

Silver refund total: 1549240.21
Order 360 refund total: 1549240.21
Selisih: 0.0


In [22]:
net_revenue_check = order_360_test[
    order_360_test["net_revenue"].round(2)
    != (
        order_360_test["gross_merchandise_value"]
        - order_360_test["discount_amount"]
        + order_360_test["shipping_revenue"]
        - order_360_test["refunded_amount"]
    ).round(2)
]

print("Net revenue mismatch:", len(net_revenue_check))

Net revenue mismatch: 0


In [5]:
orders = pd.read_sql(
    """
    SELECT *
    FROM silver.orders
    LIMIT 10
    """,
    engine
)

display(orders)

,status,order_id,store_id,customer_id,sales_channel,source_row_id,ordered_at_utc,updated_at_utc,shipping_revenue,ingested_at_utc
0,PLACED,ORD-000001,None,CUST-01757,MOBILE_APP,ORDER-ROW-0000001,2026-08-20T06:16:00Z,2026-08-20T23:16:00Z,12.0,2026-09-22 20:52:05.726606+00:00
1,RETURNED,ORD-000002,None,CUST-01160,MOBILE_APP,ORDER-ROW-0000002,2026-08-16T06:38:00Z,2026-08-17T09:38:00Z,0.0,2026-09-22 20:52:05.726606+00:00
2,PLACED,ORD-000003,None,CUST-00962,WEB,ORDER-ROW-0000003,2026-07-04T17:03:00Z,2026-07-05T16:03:00Z,7.5,2026-09-22 20:52:05.726606+00:00
3,PLACED,ORD-000004,None,CUST-00181,MARKETPLACE,ORDER-ROW-0000004,2026-08-28T13:09:00Z,2026-08-29T21:09:00Z,0.0,2026-09-22 20:52:05.726606+00:00
4,RETURNED,ORD-000005,STORE-03,CUST-02298,STORE,ORDER-ROW-0000005,2026-07-06T15:02:00Z,2026-07-07T21:02:00Z,7.5,2026-09-22 20:52:05.726606+00:00
5,CANCELLED,ORD-000006,None,CUST-00630,WEB,ORDER-ROW-0000006,2026-08-08T18:59:00Z,2026-08-09T00:59:00Z,0.0,2026-09-22 20:52:05.726606+00:00
6,FULFILLED,ORD-000007,STORE-03,CUST-00632,STORE,ORDER-ROW-0000007,2026-07-18T05:24:00Z,2026-07-19T01:24:00Z,0.0,2026-09-22 20:52:05.726606+00:00
7,PLACED,ORD-000008,None,CUST-01792,WEB,ORDER-ROW-0000008,2026-08-28T20:41:00Z,2026-08-30T03:41:00Z,0.0,2026-09-22 20:52:05.726606+00:00
8,CONFIRMED,ORD-000009,None,CUST-01220,WEB,ORDER-ROW-0000009,2026-07-15T12:02:00Z,2026-07-16T18:02:00Z,3.5,2026-09-22 20:52:05.726606+00:00
9,PLACED,ORD-000010,None,CUST-00459,MOBILE_APP,ORDER-ROW-0000010,2026-07-30T02:08:00Z,2026-07-30T15:08:00Z,0.0,2026-09-22 20:52:05.726606+00:00


In [6]:
order_items = pd.read_sql(
    """
    SELECT *
    FROM silver.order_items
    LIMIT 10
    """,
    engine
)

display(order_items)

,order_id,quantity,product_id,unit_price,order_item_id,source_row_id,item_discount_amount,ingested_at_utc
0,ORD-000001,3,PROD-00012,353.75,ITEM-000001-01,ITEM-ROW-0000001-01,0.0,2026-09-22 20:52:06.552878+00:00
1,ORD-000001,1,PROD-00073,56.99,ITEM-000001-02,ITEM-ROW-0000001-02,0.0,2026-09-22 20:52:06.552878+00:00
2,ORD-000002,2,PROD-00053,100.99,ITEM-000002-01,ITEM-ROW-0000002-01,0.0,2026-09-22 20:52:06.552878+00:00
3,ORD-000002,3,PROD-00038,433.79,ITEM-000002-02,ITEM-ROW-0000002-02,0.0,2026-09-22 20:52:06.552878+00:00
4,ORD-000003,3,PROD-00001,92.83,ITEM-000003-01,ITEM-ROW-0000003-01,5.0,2026-09-22 20:52:06.552878+00:00
5,ORD-000004,4,PROD-00047,407.20,ITEM-000004-01,ITEM-ROW-0000004-01,0.0,2026-09-22 20:52:06.552878+00:00
6,ORD-000004,3,PROD-00034,224.74,ITEM-000004-02,ITEM-ROW-0000004-02,5.0,2026-09-22 20:52:06.552878+00:00
7,ORD-000004,4,PROD-00041,23.39,ITEM-000004-03,ITEM-ROW-0000004-03,0.0,2026-09-22 20:52:06.552878+00:00
8,ORD-000005,2,PROD-00015,406.05,ITEM-000005-01,ITEM-ROW-0000005-01,0.0,2026-09-22 20:52:06.552878+00:00
9,ORD-000005,1,PROD-00045,178.03,ITEM-000005-02,ITEM-ROW-0000005-02,0.0,2026-09-22 20:52:06.552878+00:00


In [7]:
order_item_summary = pd.read_sql(
    """
    SELECT
        order_id,
        COUNT(order_item_id) AS item_count,
        SUM(quantity) AS unit_quantity,
        SUM(quantity * unit_price) AS gross_merchandise_value,
        SUM(item_discount_amount) AS discount_amount
    FROM silver.order_items
    GROUP BY order_id
    ORDER BY order_id
    LIMIT 10
    """,
    engine
)

display(order_item_summary)

,order_id,item_count,unit_quantity,gross_merchandise_value,discount_amount
0,ORD-000001,2,4.0,1118.24,0.0
1,ORD-000002,2,5.0,1503.35,0.0
2,ORD-000003,1,3.0,278.49,5.0
3,ORD-000004,3,11.0,2396.58,5.0
4,ORD-000005,4,8.0,1883.20,5.0
5,ORD-000006,4,11.0,2200.22,0.0
6,ORD-000007,4,7.0,2131.88,10.0
7,ORD-000008,3,7.0,980.93,5.0
8,ORD-000009,4,9.0,2812.32,0.0
9,ORD-000010,1,4.0,1503.56,0.0


In [8]:
order_360_test = pd.read_sql(
    """
    SELECT
        o.order_id,
        o.customer_id,
        o.ordered_at_utc AS order_date,
        o.sales_channel,
        o.store_id,
        oi.gross_merchandise_value,
        oi.discount_amount,
        o.shipping_revenue,
        oi.item_count,
        oi.unit_quantity,
        o.status AS order_status
    FROM silver.orders o
    LEFT JOIN (
        SELECT
            order_id,
            COUNT(order_item_id) AS item_count,
            SUM(quantity) AS unit_quantity,
            SUM(quantity * unit_price) AS gross_merchandise_value,
            SUM(item_discount_amount) AS discount_amount
        FROM silver.order_items
        GROUP BY order_id
    ) oi
        ON o.order_id = oi.order_id
    ORDER BY o.order_id
    LIMIT 10
    """,
    engine
)

display(order_360_test)

,order_id,customer_id,order_date,sales_channel,store_id,gross_merchandise_value,discount_amount,shipping_revenue,item_count,unit_quantity,order_status
0,ORD-000001,CUST-01757,2026-08-20T06:16:00Z,MOBILE_APP,None,1118.24,0.0,12.0,2,4.0,PLACED
1,ORD-000001,CUST-01757,2026-08-20T06:16:00Z,MOBILE_APP,None,1118.24,0.0,12.0,2,4.0,PLACED
2,ORD-000002,CUST-01160,2026-08-16T06:38:00Z,MOBILE_APP,None,1503.35,0.0,0.0,2,5.0,RETURNED
3,ORD-000003,CUST-00962,2026-07-04T17:03:00Z,WEB,None,278.49,5.0,7.5,1,3.0,PLACED
4,ORD-000004,CUST-00181,2026-08-28T13:09:00Z,MARKETPLACE,None,2396.58,5.0,0.0,3,11.0,PLACED
5,ORD-000005,CUST-02298,2026-07-06T15:02:00Z,STORE,STORE-03,1883.20,5.0,7.5,4,8.0,RETURNED
6,ORD-000006,CUST-00630,2026-08-08T18:59:00Z,WEB,None,2200.22,0.0,0.0,4,11.0,CANCELLED
7,ORD-000007,CUST-00632,2026-07-18T05:24:00Z,STORE,STORE-03,2131.88,10.0,0.0,4,7.0,FULFILLED
8,ORD-000008,CUST-01792,2026-08-28T20:41:00Z,WEB,None,980.93,5.0,0.0,3,7.0,PLACED
9,ORD-000009,CUST-01220,2026-07-15T12:02:00Z,WEB,None,2812.32,0.0,3.5,4,9.0,CONFIRMED


In [9]:
duplicates = pd.read_sql(
    """
    SELECT
        order_id,
        COUNT(*) AS row_count
    FROM silver.orders
    GROUP BY order_id
    HAVING COUNT(*) > 1
    ORDER BY row_count DESC
    """,
    engine
)

display(duplicates)

,order_id,row_count
0,ORD-000001,2


In [10]:
pd.read_sql(
    """
    SELECT *
    FROM silver.orders
    WHERE order_id = 'ORD-000001'
    """,
    engine
)

,status,order_id,store_id,customer_id,sales_channel,source_row_id,ordered_at_utc,updated_at_utc,shipping_revenue,ingested_at_utc
0,PLACED,ORD-000001,None,CUST-01757,MOBILE_APP,ORDER-ROW-0000001,2026-08-20T06:16:00Z,2026-08-20T23:16:00Z,12.0,2026-09-22 20:52:05.726606+00:00
1,PLACED,ORD-000001,None,CUST-01757,MOBILE_APP,ORDER-ROW-DUPLICATE-0001,2026-08-20T06:16:00Z,2026-08-20T23:16:00Z,12.0,2026-09-22 20:52:05.726606+00:00


In [11]:
orders_dedup = pd.read_sql(
    """
    SELECT *
    FROM (
        SELECT
            *,
            ROW_NUMBER() OVER (
                PARTITION BY order_id
                ORDER BY updated_at_utc DESC, ingested_at_utc DESC, source_row_id
            ) AS rn
        FROM silver.orders
    ) o
    WHERE rn = 1
    """,
    engine
)

print("Jumlah orders sebelum dedup :", len(orders))
print("Jumlah orders setelah dedup :", len(orders_dedup))

display(
    orders_dedup[
        orders_dedup["order_id"] == "ORD-000001"
    ]
)

Jumlah orders sebelum dedup : 10
Jumlah orders setelah dedup : 10000


,status,order_id,store_id,customer_id,sales_channel,source_row_id,ordered_at_utc,updated_at_utc,shipping_revenue,ingested_at_utc,rn
0,PLACED,ORD-000001,None,CUST-01757,MOBILE_APP,ORDER-ROW-0000001,2026-08-20T06:16:00Z,2026-08-20T23:16:00Z,12.0,2026-09-22 20:52:05.726606+00:00,1


In [13]:
order_count = pd.read_sql(
    """
    SELECT COUNT(*) AS total_orders
    FROM silver.orders
    """,
    engine
)

print("Jumlah row silver.orders :", order_count.iloc[0]["total_orders"])
print("Jumlah row setelah dedup :", len(orders_dedup))

Jumlah row silver.orders : 10001
Jumlah row setelah dedup : 10000


In [14]:
order_base = pd.read_sql(
    """
    WITH orders_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) o
        WHERE rn = 1
    ),

    order_items_summary AS (
        SELECT
            order_id,
            COUNT(order_item_id) AS item_count,
            SUM(quantity) AS unit_quantity,
            SUM(quantity * unit_price) AS gross_merchandise_value,
            SUM(item_discount_amount) AS discount_amount
        FROM silver.order_items
        GROUP BY order_id
    )

    SELECT
        o.order_id,
        o.customer_id,
        o.ordered_at_utc AS order_date,
        o.sales_channel,
        o.store_id,
        oi.gross_merchandise_value,
        oi.discount_amount,
        o.shipping_revenue,
        oi.item_count,
        oi.unit_quantity,
        o.status AS order_status

    FROM orders_dedup o

    LEFT JOIN order_items_summary oi
        ON o.order_id = oi.order_id

    ORDER BY o.order_id
    LIMIT 10
    """,
    engine
)

display(order_base)

,order_id,customer_id,order_date,sales_channel,store_id,gross_merchandise_value,discount_amount,shipping_revenue,item_count,unit_quantity,order_status
0,ORD-000001,CUST-01757,2026-08-20T06:16:00Z,MOBILE_APP,None,1118.24,0.0,12.0,2,4.0,PLACED
1,ORD-000002,CUST-01160,2026-08-16T06:38:00Z,MOBILE_APP,None,1503.35,0.0,0.0,2,5.0,RETURNED
2,ORD-000003,CUST-00962,2026-07-04T17:03:00Z,WEB,None,278.49,5.0,7.5,1,3.0,PLACED
3,ORD-000004,CUST-00181,2026-08-28T13:09:00Z,MARKETPLACE,None,2396.58,5.0,0.0,3,11.0,PLACED
4,ORD-000005,CUST-02298,2026-07-06T15:02:00Z,STORE,STORE-03,1883.20,5.0,7.5,4,8.0,RETURNED
5,ORD-000006,CUST-00630,2026-08-08T18:59:00Z,WEB,None,2200.22,0.0,0.0,4,11.0,CANCELLED
6,ORD-000007,CUST-00632,2026-07-18T05:24:00Z,STORE,STORE-03,2131.88,10.0,0.0,4,7.0,FULFILLED
7,ORD-000008,CUST-01792,2026-08-28T20:41:00Z,WEB,None,980.93,5.0,0.0,3,7.0,PLACED
8,ORD-000009,CUST-01220,2026-07-15T12:02:00Z,WEB,None,2812.32,0.0,3.5,4,9.0,CONFIRMED
9,ORD-000010,CUST-00459,2026-07-30T02:08:00Z,MOBILE_APP,None,1503.56,0.0,0.0,1,4.0,PLACED


In [15]:
payment_events = pd.read_sql(
    """
    SELECT *
    FROM silver.payment_events
    LIMIT 20
    """,
    engine
)

display(payment_events)

,event_id,event_type,ingested_at_utc,occurred_at_utc,payload.amount,payload.order_id,payload.payment_id
0,EVT-00006883,PAYMENT_CAPTURED,2026-09-22 20:52:08.632644+00:00,2026-07-19T18:14:00Z,2172.62,ORD-001121,PAY-ORD-001121
1,EVT-00010230,PAYMENT_CAPTURED,2026-09-22 20:52:08.632644+00:00,2026-09-14T02:32:00+07:00,3734.64,ORD-001662,PAY-ORD-001662
2,EVT-00051452,PAYMENT_AUTHORIZED,2026-09-22 20:52:08.632644+00:00,2026-08-08T02:02:00Z,846.12,ORD-008285,PAY-ORD-008285
3,EVT-00041158,PAYMENT_CAPTURED,2026-09-22 20:52:08.632644+00:00,2026-08-16T05:44:00Z,1785.92,ORD-006618,PAY-ORD-006618
4,EVT-00055547,PAYMENT_CAPTURED,2026-09-22 20:52:08.632644+00:00,2026-07-08T17:08:00Z,1057.10,ORD-008937,PAY-ORD-008937
5,EVT-00024618,PAYMENT_AUTHORIZED,2026-09-22 20:52:08.632644+00:00,2026-07-28T13:55:00+07:00,739.08,ORD-003970,PAY-ORD-003970
6,EVT-00048469,PAYMENT_CAPTURED,2026-09-22 20:52:08.632644+00:00,2026-08-08T09:39:00Z,1153.45,ORD-007801,PAY-ORD-007801
7,EVT-00003882,PAYMENT_AUTHORIZED,2026-09-22 20:52:08.632644+00:00,2026-06-30T05:03:00Z,1331.98,ORD-000629,PAY-ORD-000629
8,EVT-00033669,PAYMENT_AUTHORIZED,2026-09-22 20:52:08.632644+00:00,2026-09-13T11:34:00Z,1697.89,ORD-005425,PAY-ORD-005425
9,EVT-00041855,PAYMENT_AUTHORIZED,2026-09-22 20:52:08.632644+00:00,2026-08-24T14:01:00+07:00,1277.14,ORD-006728,PAY-ORD-006728


In [16]:
payment_event_types = pd.read_sql(
    """
    SELECT
        event_type,
        COUNT(*) AS event_count
    FROM silver.payment_events
    GROUP BY event_type
    ORDER BY event_type
    """,
    engine
)

display(payment_event_types)

,event_type,event_count
0,PAYMENT_AUTHORIZED,10302
1,PAYMENT_CAPTURED,11030
2,PAYMENT_FAILED,1250


In [18]:
payment_summary = pd.read_sql(
    """
    WITH latest_payment AS (
        SELECT
            "payload.order_id" AS order_id,
            event_type,
            occurred_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY "payload.order_id"
                ORDER BY occurred_at_utc DESC, event_id DESC
            ) AS rn
        FROM silver.payment_events
    ),

    captured_payment AS (
        SELECT
            "payload.order_id" AS order_id,
            SUM("payload.amount") AS captured_payment_amount
        FROM silver.payment_events
        WHERE event_type = 'PAYMENT_CAPTURED'
        GROUP BY "payload.order_id"
    )

    SELECT
        lp.order_id,
        cp.captured_payment_amount,
        lp.event_type AS payment_status,
        lp.occurred_at_utc AS latest_payment_event_at
    FROM latest_payment lp
    LEFT JOIN captured_payment cp
        ON lp.order_id = cp.order_id
    WHERE lp.rn = 1
    ORDER BY lp.order_id
    LIMIT 10
    """,
    engine
)

display(payment_summary)

,order_id,captured_payment_amount,payment_status,latest_payment_event_at
0,ORD-000001,1125.24,PAYMENT_CAPTURED,2026-08-20T07:16:00Z
1,ORD-000002,1490.35,PAYMENT_CAPTURED,2026-08-16T07:38:00Z
2,ORD-000003,270.99,PAYMENT_AUTHORIZED,2026-07-05T00:08:00+07:00
3,ORD-000004,2361.58,PAYMENT_CAPTURED,2026-08-28T14:09:00Z
4,ORD-000005,1867.70,PAYMENT_AUTHORIZED,2026-07-06T22:07:00+07:00
5,ORD-000006,NaN,PAYMENT_FAILED,2026-08-08T19:59:00Z
6,ORD-000007,2482.04,PAYMENT_CAPTURED,2026-07-18T13:24:00+07:00
7,ORD-000008,955.93,PAYMENT_CAPTURED,2026-08-28T21:41:00Z
8,ORD-000009,2800.82,PAYMENT_CAPTURED,2026-07-15T13:02:00Z
9,ORD-000010,1473.56,PAYMENT_CAPTURED,2026-07-30T03:08:00Z


In [19]:
pd.read_sql(
    """
    SELECT
        event_id,
        event_type,
        "payload.amount" AS amount,
        "payload.order_id" AS order_id,
        occurred_at_utc
    FROM silver.payment_events
    WHERE "payload.order_id" = 'ORD-000003'
    ORDER BY occurred_at_utc
    """,
    engine
)

,event_id,event_type,amount,order_id,occurred_at_utc
0,EVT-00000012,PAYMENT_CAPTURED,270.99,ORD-000003,2026-07-04T18:03:00Z
1,EVT-00000011,PAYMENT_AUTHORIZED,270.99,ORD-000003,2026-07-05T00:08:00+07:00


In [20]:
pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        COUNT(*) AS captured_event_count,
        SUM("payload.amount") AS captured_amount
    FROM silver.payment_events
    WHERE event_type = 'PAYMENT_CAPTURED'
    GROUP BY "payload.order_id"
    HAVING COUNT(*) > 1
    ORDER BY captured_event_count DESC
    LIMIT 10
    """,
    engine
)

,order_id,captured_event_count,captured_amount
0,ORD-007005,3,1531.01
1,ORD-008160,3,3170.16
2,ORD-001416,3,1445.43
3,ORD-005263,3,2735.92
4,ORD-009521,3,3890.35
5,ORD-003015,3,3043.64
6,ORD-001949,3,1967.76
7,ORD-004441,3,2498.42
8,ORD-000303,3,2217.12
9,ORD-004373,3,694.14


In [21]:
refund_events = pd.read_sql(
    """
    SELECT *
    FROM silver.refund_events
    LIMIT 10
    """,
    engine
)

display(refund_events)

,event_id,event_type,ingested_at_utc,occurred_at_utc,payload.amount,payload.order_id,payload.refund_id
0,EVT-00040083,REFUND_ISSUED,2026-09-22 20:52:09.697798+00:00,2026-06-25T16:16:00Z,425.51,ORD-006441,REF-ORD-006441-01
1,EVT-00052826,REFUND_COMPLETED,2026-09-22 20:52:09.697798+00:00,2026-07-08T11:36:00Z,1623.72,ORD-008503,REF-ORD-008503-01
2,EVT-00010727,REFUND_ISSUED,2026-09-22 20:52:09.697798+00:00,2026-09-17T01:48:00Z,214.32,ORD-001742,REF-ORD-001742-01
3,EVT-00059354,REFUND_COMPLETED,2026-09-22 20:52:09.697798+00:00,2026-09-09T09:06:00Z,114.34,ORD-009547,REF-ORD-009547-01
4,EVT-00000252,REFUND_ISSUED,2026-09-22 20:52:09.697798+00:00,2026-08-14T22:02:00Z,340.56,ORD-000044,REF-ORD-000044-01
5,EVT-00058621,REFUND_ISSUED,2026-09-22 20:52:09.697798+00:00,2026-08-08T09:58:00Z,1651.40,ORD-009431,REF-ORD-009431-01
6,EVT-00019123,REFUND_COMPLETED,2026-09-22 20:52:09.697798+00:00,2026-07-15T18:13:00Z,2197.38,ORD-003092,REF-ORD-003092-01
7,EVT-00038878,REFUND_COMPLETED,2026-09-22 20:52:09.697798+00:00,2026-08-06T17:55:00Z,1312.54,ORD-006250,REF-ORD-006250-01
8,EVT-00012674,REFUND_ISSUED,2026-09-22 20:52:09.697798+00:00,2026-08-08T09:03:00Z,574.98,ORD-002054,REF-ORD-002054-01
9,EVT-00030956,REFUND_ISSUED,2026-09-22 20:52:09.697798+00:00,2026-08-31T05:27:00Z,1195.38,ORD-004987,REF-ORD-004987-01


In [22]:
pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        "payload.refund_id" AS refund_id,
        event_type,
        "payload.amount" AS amount,
        occurred_at_utc
    FROM silver.refund_events
    WHERE "payload.order_id" = 'ORD-000044'
    ORDER BY occurred_at_utc
    """,
    engine
)

,order_id,refund_id,event_type,amount,occurred_at_utc
0,ORD-000044,REF-ORD-000044-01,REFUND_ISSUED,340.56,2026-08-14T22:02:00Z
1,ORD-000044,REF-ORD-000044-01,REFUND_COMPLETED,340.56,2026-08-16T05:02:00+07:00


In [24]:
pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        COUNT(DISTINCT "payload.refund_id") AS refund_count,
        SUM("payload.amount") AS refunded_amount
    FROM silver.refund_events
    WHERE event_type = 'REFUND_COMPLETED'
    GROUP BY "payload.order_id"
    HAVING COUNT(DISTINCT "payload.refund_id") > 1
    ORDER BY refund_count DESC
    LIMIT 10
    """,
    engine
)

,order_id,refund_count,refunded_amount


In [25]:
refund_summary = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        SUM("payload.amount") AS refunded_amount
    FROM silver.refund_events
    WHERE event_type = 'REFUND_COMPLETED'
    GROUP BY "payload.order_id"
    ORDER BY "payload.order_id"
    LIMIT 10
    """,
    engine
)

display(refund_summary)

,order_id,refunded_amount
0,ORD-000018,1063.59
1,ORD-000035,432.69
2,ORD-000038,275.21
3,ORD-000039,141.39
4,ORD-000044,340.56
5,ORD-000052,1260.65
6,ORD-000053,288.59
7,ORD-000058,3203.06
8,ORD-000066,299.07
9,ORD-000069,1072.71


In [26]:
return_events = pd.read_sql(
    """
    SELECT *
    FROM silver.return_events
    LIMIT 10
    """,
    engine
)

display(return_events)

,event_id,event_type,ingested_at_utc,occurred_at_utc,payload.order_id,payload.return_id
0,EVT-00036639,RETURN_CLOSED,2026-09-22 20:52:09.935900+00:00,2026-08-22T01:10:00Z,ORD-005899,RETURN-ORD-005899
1,EVT-00030743,RETURN_REQUESTED,2026-09-22 20:52:09.935900+00:00,2026-07-03T16:01:00Z,ORD-004953,RETURN-ORD-004953
2,EVT-00027189,RETURN_RECEIVED,2026-09-22 20:52:09.935900+00:00,2026-09-23T14:34:00Z,ORD-004382,RETURN-ORD-004382
3,EVT-00050012,RETURN_REQUESTED,2026-09-22 20:52:09.935900+00:00,2026-07-12T08:31:00Z,ORD-008051,RETURN-ORD-008051
4,EVT-00038062,RETURN_RECEIVED,2026-09-22 20:52:09.935900+00:00,2026-09-15T16:05:00Z,ORD-006125,RETURN-ORD-006125
5,EVT-00001015,RETURN_REQUESTED,2026-09-22 20:52:09.935900+00:00,2026-08-02T17:47:00Z,ORD-000169,RETURN-ORD-000169
6,EVT-00033233,RETURN_CLOSED,2026-09-22 20:52:09.935900+00:00,2026-09-17T04:07:00Z,ORD-005354,RETURN-ORD-005354
7,EVT-00060351,RETURN_RECEIVED,2026-09-22 20:52:09.935900+00:00,2026-08-24T04:37:00Z,ORD-009706,RETURN-ORD-009706
8,EVT-00035366,RETURN_RECEIVED,2026-09-22 20:52:09.935900+00:00,2026-09-26T09:40:00Z,ORD-005694,RETURN-ORD-005694
9,EVT-00034821,RETURN_REQUESTED,2026-09-22 20:52:09.935900+00:00,2026-09-15T22:24:00Z,ORD-005606,RETURN-ORD-005606


In [27]:
return_event_types = pd.read_sql(
    """
    SELECT
        event_type,
        COUNT(*) AS event_count
    FROM silver.return_events
    GROUP BY event_type
    ORDER BY event_type
    """,
    engine
)

display(return_event_types)

,event_type,event_count
0,RETURN_CLOSED,976
1,RETURN_RECEIVED,1156
2,RETURN_REQUESTED,1421


In [28]:
pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        "payload.return_id" AS return_id,
        event_type,
        occurred_at_utc
    FROM silver.return_events
    WHERE "payload.return_id" = 'RETURN-ORD-005899'
    ORDER BY occurred_at_utc
    """,
    engine
)

,order_id,return_id,event_type,occurred_at_utc
0,ORD-005899,RETURN-ORD-005899,RETURN_REQUESTED,2026-08-17T01:10:00Z
1,ORD-005899,RETURN-ORD-005899,RETURN_RECEIVED,2026-08-20T01:10:00Z
2,ORD-005899,RETURN-ORD-005899,RETURN_CLOSED,2026-08-22T01:10:00Z


In [29]:
return_summary = pd.read_sql(
    """
    SELECT
        order_id,
        event_type AS return_status,
        occurred_at_utc AS return_status_at
    FROM (
        SELECT
            "payload.order_id" AS order_id,
            event_type,
            occurred_at_utc,
            ROW_NUMBER() OVER (
                PARTITION BY "payload.order_id"
                ORDER BY occurred_at_utc DESC, event_id DESC
            ) AS rn
        FROM silver.return_events
    ) r
    WHERE rn = 1
    ORDER BY order_id
    LIMIT 10
    """,
    engine
)

display(return_summary)

,order_id,return_status,return_status_at
0,ORD-000003,RETURN_REQUESTED,2026-07-09T17:03:00Z
1,ORD-000011,RETURN_CLOSED,2026-07-04T14:37:00Z
2,ORD-000016,RETURN_RECEIVED,2026-07-17T13:26:00Z
3,ORD-000029,RETURN_CLOSED,2026-08-06T18:32:00Z
4,ORD-000034,RETURN_RECEIVED,2026-09-02T17:54:00Z
5,ORD-000039,RETURN_CLOSED,2026-07-24T14:14:00Z
6,ORD-000043,RETURN_CLOSED,2026-09-04T22:43:00Z
7,ORD-000044,RETURN_CLOSED,2026-08-21T22:02:00Z
8,ORD-000049,RETURN_REQUESTED,2026-08-29T09:12:00Z
9,ORD-000056,RETURN_REQUESTED,2026-08-26T10:38:00Z


In [30]:
order_promotions = pd.read_sql(
    """
    SELECT *
    FROM silver.order_promotions
    LIMIT 10
    """,
    engine
)

display(order_promotions)

,order_id,promotion_id,source_row_id,discount_amount,ingested_at_utc
0,ORD-000001,PROMO-006,ORDER-PROMO-ROW-0000001-PROMO-006,5,2026-09-22 20:52:07.701843+00:00
1,ORD-000002,PROMO-006,ORDER-PROMO-ROW-0000002-PROMO-006,10,2026-09-22 20:52:07.701843+00:00
2,ORD-000002,PROMO-007,ORDER-PROMO-ROW-0000002-PROMO-007,3,2026-09-22 20:52:07.701843+00:00
3,ORD-000003,PROMO-012,ORDER-PROMO-ROW-0000003-PROMO-012,10,2026-09-22 20:52:07.701843+00:00
4,ORD-000004,PROMO-004,ORDER-PROMO-ROW-0000004-PROMO-004,15,2026-09-22 20:52:07.701843+00:00
5,ORD-000004,PROMO-008,ORDER-PROMO-ROW-0000004-PROMO-008,15,2026-09-22 20:52:07.701843+00:00
6,ORD-000005,PROMO-008,ORDER-PROMO-ROW-0000005-PROMO-008,3,2026-09-22 20:52:07.701843+00:00
7,ORD-000005,PROMO-012,ORDER-PROMO-ROW-0000005-PROMO-012,15,2026-09-22 20:52:07.701843+00:00
8,ORD-000006,PROMO-008,ORDER-PROMO-ROW-0000006-PROMO-008,3,2026-09-22 20:52:07.701843+00:00
9,ORD-000007,PROMO-003,ORDER-PROMO-ROW-0000007-PROMO-003,5,2026-09-22 20:52:07.701843+00:00


In [31]:
promotion_summary = pd.read_sql(
    """
    SELECT
        order_id,
        SUM(discount_amount) AS promotion_discount,
        COUNT(DISTINCT promotion_id) AS promotion_count
    FROM silver.order_promotions
    GROUP BY order_id
    ORDER BY order_id
    LIMIT 10
    """,
    engine
)

display(promotion_summary)

,order_id,promotion_discount,promotion_count
0,ORD-000001,5.0,1
1,ORD-000002,13.0,2
2,ORD-000003,10.0,1
3,ORD-000004,30.0,2
4,ORD-000005,18.0,2
5,ORD-000006,3.0,1
6,ORD-000007,20.0,2
7,ORD-000008,20.0,2
8,ORD-000009,15.0,1
9,ORD-000010,30.0,2


In [3]:
order_360_test = pd.read_sql(
    """
    WITH orders_dedup AS (
        SELECT *
        FROM (
            SELECT
                *,
                ROW_NUMBER() OVER (
                    PARTITION BY order_id
                    ORDER BY updated_at_utc DESC,
                             ingested_at_utc DESC,
                             source_row_id
                ) AS rn
            FROM silver.orders
        ) o
        WHERE rn = 1
    ),

    order_items_summary AS (
        SELECT
            order_id,
            COUNT(order_item_id) AS item_count,
            SUM(quantity) AS unit_quantity,
            SUM(quantity * unit_price) AS gross_merchandise_value,
            SUM(item_discount_amount) AS discount_amount
        FROM silver.order_items
        GROUP BY order_id
    ),

    payment_summary AS (
        SELECT
            "payload.order_id" AS order_id,
            SUM("payload.amount") AS captured_payment_amount
        FROM silver.payment_events
        WHERE event_type = 'PAYMENT_CAPTURED'
        GROUP BY "payload.order_id"
    ),

    refund_summary AS (
        SELECT
            "payload.order_id" AS order_id,
            SUM("payload.amount") AS refunded_amount
        FROM silver.refund_events
        WHERE event_type = 'REFUND_COMPLETED'
        GROUP BY "payload.order_id"
    ),

    return_summary AS (
        SELECT
            order_id,
            event_type AS return_status
        FROM (
            SELECT
                "payload.order_id" AS order_id,
                event_type,
                occurred_at_utc,
                ROW_NUMBER() OVER (
                    PARTITION BY "payload.order_id"
                    ORDER BY occurred_at_utc DESC, event_id DESC
                ) AS rn
            FROM silver.return_events
        ) r
        WHERE rn = 1
    ),

    promotion_summary AS (
        SELECT
            order_id,
            SUM(discount_amount) AS promotion_discount,
            COUNT(DISTINCT promotion_id) AS promotion_count
        FROM silver.order_promotions
        GROUP BY order_id
    )

    SELECT
        o.order_id,
        o.customer_id,
        o.ordered_at_utc AS order_date,
        o.sales_channel,
        o.store_id,

        oi.gross_merchandise_value,
        oi.discount_amount,
        p.promotion_discount,
        o.shipping_revenue,

        oi.item_count,
        oi.unit_quantity,
        p.promotion_count,

        pay.captured_payment_amount,
        r.refunded_amount,

        o.status AS order_status,
        ret.return_status

    FROM orders_dedup o

    LEFT JOIN order_items_summary oi
        ON o.order_id = oi.order_id

    LEFT JOIN payment_summary pay
        ON o.order_id = pay.order_id

    LEFT JOIN refund_summary r
        ON o.order_id = r.order_id

    LEFT JOIN return_summary ret
        ON o.order_id = ret.order_id

    LEFT JOIN promotion_summary p
        ON o.order_id = p.order_id

    ORDER BY o.order_id
    LIMIT 10
    """,
    engine
)

display(order_360_test)

,order_id,customer_id,order_date,sales_channel,store_id,gross_merchandise_value,discount_amount,promotion_discount,shipping_revenue,item_count,unit_quantity,promotion_count,captured_payment_amount,refunded_amount,order_status,return_status
0,ORD-000001,CUST-01757,2026-08-20T06:16:00Z,MOBILE_APP,None,1118.24,0.0,5.0,12.0,2,4.0,1,1125.24,None,PLACED,None
1,ORD-000002,CUST-01160,2026-08-16T06:38:00Z,MOBILE_APP,None,1503.35,0.0,13.0,0.0,2,5.0,2,1490.35,None,RETURNED,None
2,ORD-000003,CUST-00962,2026-07-04T17:03:00Z,WEB,None,278.49,5.0,10.0,7.5,1,3.0,1,270.99,None,PLACED,RETURN_REQUESTED
3,ORD-000004,CUST-00181,2026-08-28T13:09:00Z,MARKETPLACE,None,2396.58,5.0,30.0,0.0,3,11.0,2,2361.58,None,PLACED,None
4,ORD-000005,CUST-02298,2026-07-06T15:02:00Z,STORE,STORE-03,1883.20,5.0,18.0,7.5,4,8.0,2,1867.70,None,RETURNED,None
5,ORD-000006,CUST-00630,2026-08-08T18:59:00Z,WEB,None,2200.22,0.0,3.0,0.0,4,11.0,1,NaN,None,CANCELLED,None
6,ORD-000007,CUST-00632,2026-07-18T05:24:00Z,STORE,STORE-03,2131.88,10.0,20.0,0.0,4,7.0,2,2482.04,None,FULFILLED,None
7,ORD-000008,CUST-01792,2026-08-28T20:41:00Z,WEB,None,980.93,5.0,20.0,0.0,3,7.0,2,955.93,None,PLACED,None
8,ORD-000009,CUST-01220,2026-07-15T12:02:00Z,WEB,None,2812.32,0.0,15.0,3.5,4,9.0,1,2800.82,None,CONFIRMED,None
9,ORD-000010,CUST-00459,2026-07-30T02:08:00Z,MOBILE_APP,None,1503.56,0.0,30.0,0.0,1,4.0,2,1473.56,None,PLACED,None


In [33]:
order_360_test.groupby("order_id").size().sort_values(ascending=False).head(10)

order_id
ORD-000001    1
ORD-000002    1
ORD-000003    1
ORD-000004    1
ORD-000005    1
ORD-000006    1
ORD-000007    1
ORD-000008    1
ORD-000009    1
ORD-000010    1
dtype: int64

In [34]:
order_360_test["net_revenue"] = (
    order_360_test["gross_merchandise_value"]
    - order_360_test["discount_amount"]
    + order_360_test["shipping_revenue"]
    - order_360_test["refunded_amount"].fillna(0)
)

display(
    order_360_test[
        [
            "order_id",
            "gross_merchandise_value",
            "discount_amount",
            "shipping_revenue",
            "refunded_amount",
            "net_revenue"
        ]
    ].head(10)
)

C:\Users\user\AppData\Local\Temp\ipykernel_30336\1394781016.py:5: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  - order_360_test["refunded_amount"].fillna(0)


,order_id,gross_merchandise_value,discount_amount,shipping_revenue,refunded_amount,net_revenue
0,ORD-000001,1118.24,0.0,12.0,None,1130.24
1,ORD-000002,1503.35,0.0,0.0,None,1503.35
2,ORD-000003,278.49,5.0,7.5,None,280.99
3,ORD-000004,2396.58,5.0,0.0,None,2391.58
4,ORD-000005,1883.20,5.0,7.5,None,1885.70
5,ORD-000006,2200.22,0.0,0.0,None,2200.22
6,ORD-000007,2131.88,10.0,0.0,None,2121.88
7,ORD-000008,980.93,5.0,0.0,None,975.93
8,ORD-000009,2812.32,0.0,3.5,None,2815.82
9,ORD-000010,1503.56,0.0,0.0,None,1503.56


In [35]:
pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        "payload.payment_id" AS payment_id,
        event_type,
        "payload.amount" AS amount,
        occurred_at_utc
    FROM silver.payment_events
    WHERE "payload.order_id" = 'ORD-000001'
    ORDER BY occurred_at_utc
    """,
    engine
)

,order_id,payment_id,event_type,amount,occurred_at_utc
0,ORD-000001,PAY-ORD-000001,PAYMENT_AUTHORIZED,1125.24,2026-08-20T06:21:00Z
1,ORD-000001,PAY-ORD-000001,PAYMENT_CAPTURED,1125.24,2026-08-20T07:16:00Z


In [36]:
payment_patterns = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,
        STRING_AGG(
            DISTINCT event_type,
            ', ' ORDER BY event_type
        ) AS payment_events
    FROM silver.payment_events
    GROUP BY "payload.order_id"
    ORDER BY order_id
    LIMIT 20
    """,
    engine
)

display(payment_patterns)

,order_id,payment_events
0,ORD-000001,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"
1,ORD-000002,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"
2,ORD-000003,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"
3,ORD-000004,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"
4,ORD-000005,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"
5,ORD-000006,"PAYMENT_AUTHORIZED, PAYMENT_FAILED"
6,ORD-000007,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"
7,ORD-000008,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"
8,ORD-000009,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"
9,ORD-000010,"PAYMENT_AUTHORIZED, PAYMENT_CAPTURED"


In [37]:
payment_status_test = pd.read_sql(
    """
    SELECT
        "payload.order_id" AS order_id,

        CASE
            WHEN BOOL_OR(event_type = 'PAYMENT_CAPTURED')
                THEN 'PAYMENT_CAPTURED'

            WHEN BOOL_OR(event_type = 'PAYMENT_FAILED')
                THEN 'PAYMENT_FAILED'

            WHEN BOOL_OR(event_type = 'PAYMENT_AUTHORIZED')
                THEN 'PAYMENT_AUTHORIZED'

            ELSE NULL
        END AS payment_status

    FROM silver.payment_events
    GROUP BY "payload.order_id"
    ORDER BY "payload.order_id"
    LIMIT 20
    """,
    engine
)

display(payment_status_test)

,order_id,payment_status
0,ORD-000001,PAYMENT_CAPTURED
1,ORD-000002,PAYMENT_CAPTURED
2,ORD-000003,PAYMENT_CAPTURED
3,ORD-000004,PAYMENT_CAPTURED
4,ORD-000005,PAYMENT_CAPTURED
5,ORD-000006,PAYMENT_FAILED
6,ORD-000007,PAYMENT_CAPTURED
7,ORD-000008,PAYMENT_CAPTURED
8,ORD-000009,PAYMENT_CAPTURED
9,ORD-000010,PAYMENT_CAPTURED


In [1]:
customer_tables = pd.read_sql(
    """
    SELECT
        table_name,
        column_name,
        data_type
    FROM information_schema.columns
    WHERE table_schema = 'silver'
      AND table_name IN (
          'customers',
          'customer_profiles',
          'customer_addresses'
      )
    ORDER BY table_name, ordinal_position
    """,
    engine
)

display(customer_tables)

NameError: name 'pd' is not defined